# Limpieza del Censo 2024: comuna de Estación Central

Objetivo: quedarme solo con la comuna de Estación Central y con las variables que sirven para el cálculo de la huella de carbono comunal.

**Importante:** el archivo es la cartografía *zonal* del Censo 2024 (región 13). Cada fila es una **zona censal** con conteos (personas, viviendas, hogares, etc.), no una persona individual. El total de personas de la comuna sale de sumar la columna de personas de todas sus zonas.

## 1. Librerías

In [1]:
import pandas as pd
import geopandas as gpd

## 2. Carga de la base

In [2]:
RUTA_CENSO_ZONAL = "../IDG/Parquet/Cartografía_censo2024_R13_Zonal.parquet"

censo_zonal = gpd.read_parquet(RUTA_CENSO_ZONAL)

print("Filas (zonas) y columnas:", censo_zonal.shape)
print("CRS:", censo_zonal.crs)
print("Columna de geometría:", censo_zonal.geometry.name)
censo_zonal.head(3)

Filas (zonas) y columnas: (1865, 213)
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "SIRGAS 2000", "datum": {"type": "GeodeticReferenceFrame", "name": "Sistema de Referencia Geocentrico para las AmericaS 2000", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Horizontal component of 3D system.", "area": "Latin America - Central America and South America - onshore and offshore. Brazil - onshore and offshore.", "bbox": {"south_latitude": -59.87, "west_longitude": -122.19, "north_latitude": 32.72, "east_longitude": -25.28}, "id": {"authority": "EPSG", "code": 4674}}
Columna de geometría: SHAPE


,OBJECTID,CUT,COD_REGION,REGION,COD_PROVINCIA,PROVINCIA,COMUNA,AREA_C,DISTRITO,COD_DISTRITO,...,n_fuente_elect_otro,n_fuente_elect_no_tiene,n_basura_servicios,n_basura_entierra,n_basura_eriazo,n_basura_rio,n_basura_otro,SHAPE_Length,SHAPE_Area,SHAPE
0,2663,13402,13,METROPOLITANA DE SANTIAGO,134,MAIPO,BUIN,URBANO,EL RECURSO,6,...,4.0,1.0,704.0,3.0,0.0,0.0,4.0,0.085226,0.000076,"MULTIPOLYGON (((-70.66374 -33.66766, -70.66357..."
1,2664,13402,13,METROPOLITANA DE SANTIAGO,134,MAIPO,BUIN,URBANO,ALTO JAHUEL,2,...,1.0,0.0,895.0,0.0,1.0,0.0,2.0,0.083812,0.000099,"MULTIPOLYGON (((-70.68434 -33.73325, -70.68432..."
2,2665,13402,13,METROPOLITANA DE SANTIAGO,134,MAIPO,BUIN,URBANO,VALDIVIA DE PAINE,5,...,6.0,4.0,1338.0,2.0,1.0,0.0,1.0,0.176524,0.000170,"MULTIPOLYGON (((-70.79271 -33.77484, -70.79476..."


## 3. Revisar las columnas (tipo, datos no nulos y un ejemplo)

In [3]:
sin_geom = censo_zonal.drop(columns=censo_zonal.geometry.name)

resumen_columnas = pd.DataFrame({
    "dtype": sin_geom.dtypes.astype(str),
    "no_nulos": sin_geom.notna().sum(),
    "ejemplo": sin_geom.iloc[0],
})
pd.set_option("display.max_rows", None)
resumen_columnas

,dtype,no_nulos,ejemplo
OBJECTID,int64,1865,2663
CUT,int32,1865,13402
COD_REGION,int32,1865,13
REGION,object,1865,METROPOLITANA DE SANTIAGO
COD_PROVINCIA,int32,1865,134
PROVINCIA,object,1865,MAIPO
COMUNA,object,1865,BUIN
AREA_C,object,1865,URBANO
DISTRITO,object,1865,EL RECURSO
COD_DISTRITO,int16,1865,6


## 4. Filtrar Estación Central

Ya comprobé con `censo_zonal["COMUNA"].unique()` que la comuna se llama `ESTACIÓN CENTRAL`, y la columna `CUT` trae el código oficial 13106.

In [4]:
columna_comuna = "COMUNA"

ec = censo_zonal[censo_zonal[columna_comuna] == "ESTACIÓN CENTRAL"].copy()

# Control: el código oficial (CUT 13106) debe marcar exactamente las mismas zonas.
mismas_zonas = censo_zonal["CUT"].astype(str).str.replace(r"\.0$", "", regex=True) == "13106"
print("Zonas con CUT = 13106:", mismas_zonas.sum())
print("Zonas con COMUNA = ESTACIÓN CENTRAL:", len(ec))
print("Las zonas filtradas por nombre tienen todas el CUT 13106:", mismas_zonas[ec.index].all())

assert mismas_zonas.sum() == len(ec) and mismas_zonas[ec.index].all(), "CUT y COMUNA no coinciden"

print("Zonas de Estación Central:", len(ec), "de", len(censo_zonal))
print("Geometrías válidas:", ec.geometry.is_valid.all())
print("Límites (minx, miny, maxx, maxy):", ec.total_bounds)

Zonas con CUT = 13106: 43
Zonas con COMUNA = ESTACIÓN CENTRAL: 43
Las zonas filtradas por nombre tienen todas el CUT 13106: True
Zonas de Estación Central: 43 de 1865
Geometrías válidas: True
Límites (minx, miny, maxx, maxy): [-70.73375795 -33.48136219 -70.67169085 -33.44384312]


## 5. Buscar las variables útiles para el estudio

La lista agrupa las columnas por tema según su nombre. Es solo una guía: hay que mirar cuáles son realmente las que necesito.

In [5]:
TEMAS = {
    "identificadores y territorio": ["ZONA", "MANZ", "DISTR", "LOCAL", "ID", "COD", "NOM", "AREA", "SUPERF"],
    "personas": ["PER", "POBL", "HAB", "SEXO", "EDAD", "HOMBRE", "MUJER"],
    "viviendas": ["VIV"],
    "hogares": ["HOG"],
    "energía, agua y residuos": ["ENER", "COMB", "CALEF", "COCIN", "GAS", "LENA", "ELEC", "AGUA", "BASURA", "RESID"],
    "transporte y traslado": ["TRANS", "TRASL", "MOVIL", "AUTO", "VEHIC", "BICI"],
}

for tema, palabras in TEMAS.items():
    cols = [c for c in sin_geom.columns if any(p in c.upper() for p in palabras)]
    print(f"\n== {tema} ({len(cols)} columnas)")
    for c in cols:
        if pd.api.types.is_numeric_dtype(ec[c]):
            print(f"   {c:40s} numérica, suma en Estación Central = {ec[c].sum():,.0f}")
        else:
            print(f"   {c:40s} texto/otro, ejemplo = {ec[c].iloc[0]}")


== identificadores y territorio (28 columnas)
   OBJECTID                                 numérica, suma en Estación Central = 133,515
   COD_REGION                               numérica, suma en Estación Central = 559
   COD_PROVINCIA                            numérica, suma en Estación Central = 5,633
   AREA_C                                   texto/otro, ejemplo = URBANO
   DISTRITO                                 texto/otro, ejemplo = ECUADOR
   COD_DISTRITO                             numérica, suma en Estación Central = 346
   COD_LOCALIDAD                            numérica, suma en Estación Central = 43
   COD_ZONA                                 numérica, suma en Estación Central = 100
   LOCALIDAD                                texto/otro, ejemplo = GRAN SANTIAGO
   COD_ENTIDAD                              numérica, suma en Estación Central = 43
   ENTIDAD                                  texto/otro, ejemplo = ESTACIÓN CENTRAL
   COD_CATEGORIA                            

## 6. Dejar solo las variables elegidas

Elegí las variables según lo que pide la calculadora de Huella Chile y lo que sirve para repartir la información regional a la comuna:

- **Población y hogares:** total de personas, hogares y viviendas (la población es el dato que pide la calculadora al crear el inventario).
- **Energía residencial:** combustible para cocinar y para calefacción, y fuente de electricidad.
- **Residuos:** cómo eliminan la basura los hogares.
- **Aguas residuales:** tipo de servicio higiénico (alcantarillado, fosa, pozo, etc.).
- **Transporte:** medio de transporte usado por la población.

No incluí edad, educación, materiales de la vivienda, internet ni rama de actividad (CAENES), porque no se usan en el cálculo. La rama de actividad se refiere al empleo de los residentes y no a dónde están las empresas, así que no sirve para el catastro industrial.

In [6]:
ID_COLS = ["CUT", "COMUNA", "AREA_C", "DISTRITO", "COD_DISTRITO", "COD_ZONA", "ID_ZONA"]

VARIABLES_ESTUDIO = [
    # Población y hogares
    "n_per", "n_hombres", "n_mujeres", "n_hog", "prom_per_hog",
    # Viviendas
    "n_vp", "n_vp_ocupada", "n_vp_desocupada",
    "n_tipo_viv_casa", "n_tipo_viv_depto", "n_tipo_viv_indigena", "n_tipo_viv_pieza",
    "n_tipo_viv_mediagua", "n_tipo_viv_movil", "n_tipo_viv_otro",
    # Combustible para cocinar
    "n_comb_cocina_gas", "n_comb_cocina_parafina", "n_comb_cocina_lena", "n_comb_cocina_pellet",
    "n_comb_cocina_carbon", "n_comb_cocina_electricidad", "n_comb_cocina_solar", "n_comb_cocina_no_utiliza",
    # Combustible para calefacción
    "n_comb_calefaccion_gas", "n_comb_calefaccion_parafina", "n_comb_calefaccion_lena",
    "n_comb_calefaccion_pellet", "n_comb_calefaccion_carbon", "n_comb_calefaccion_electricidad",
    "n_comb_calefaccion_otra", "n_comb_calefaccion_no_utiliza",
    # Fuente de electricidad
    "n_fuente_elect_publica", "n_fuente_elect_diesel", "n_fuente_elect_solar",
    "n_fuente_elect_eolica", "n_fuente_elect_otro", "n_fuente_elect_no_tiene",
    # Eliminación de basura
    "n_basura_servicios", "n_basura_entierra", "n_basura_eriazo", "n_basura_rio", "n_basura_otro",
    # Servicio higiénico (aguas residuales)
    "n_serv_hig_alc_dentro", "n_serv_hig_alc_fuera", "n_serv_hig_fosa", "n_serv_hig_pozo",
    "n_serv_hig_acequia_canal", "n_serv_hig_cajon_otro", "n_serv_hig_bano_quimico",
    "n_serv_hig_bano_seco", "n_serv_hig_no_tiene",
    # Medio de transporte
    "n_transporte_auto", "n_transporte_publico", "n_transporte_camina", "n_transporte_bicicleta",
    "n_transporte_motocicleta", "n_transporte_cab_lan_bote", "n_transporte_otros",
]

cols_finales = ID_COLS + [c for c in VARIABLES_ESTUDIO if c not in ID_COLS] + [ec.geometry.name]

faltan = [c for c in cols_finales if c not in ec.columns]
assert not faltan, f"Estas columnas no existen en la base: {faltan}"

ec_estudio = ec[cols_finales].copy()
print("Filas:", len(ec_estudio), "| Columnas:", len(ec_estudio.columns))
ec_estudio.head()

Filas: 43 | Columnas: 66


,CUT,COMUNA,AREA_C,DISTRITO,COD_DISTRITO,COD_ZONA,ID_ZONA,n_per,n_hombres,n_mujeres,...,n_serv_hig_bano_seco,n_serv_hig_no_tiene,n_transporte_auto,n_transporte_publico,n_transporte_camina,n_transporte_bicicleta,n_transporte_motocicleta,n_transporte_cab_lan_bote,n_transporte_otros,SHAPE
420,13106,ESTACIÓN CENTRAL,URBANO,ECUADOR,11,3,1.310611e+09,3955.0,1929.0,2026.0,...,0.0,0.0,304.0,1319.0,195.0,49.0,51.0,0.0,57.0,"MULTIPOLYGON (((-70.70226 -33.45458, -70.70188..."
421,13106,ESTACIÓN CENTRAL,URBANO,ECUADOR,11,1,1.310611e+09,1233.0,599.0,634.0,...,0.0,0.0,106.0,336.0,40.0,18.0,14.0,0.0,18.0,"MULTIPOLYGON (((-70.70637 -33.45207, -70.70636..."
422,13106,ESTACIÓN CENTRAL,URBANO,BERNAL DEL MERCADO,13,4,1.310613e+09,1124.0,568.0,556.0,...,0.0,0.0,57.0,413.0,112.0,5.0,6.0,0.0,12.0,"MULTIPOLYGON (((-70.6805 -33.45107, -70.68058 ..."
423,13106,ESTACIÓN CENTRAL,URBANO,PILA DEL GANSO,10,2,1.310610e+09,16251.0,7962.0,8289.0,...,0.0,1.0,973.0,7416.0,558.0,135.0,246.0,0.0,415.0,"MULTIPOLYGON (((-70.70556 -33.45716, -70.70554..."
424,13106,ESTACIÓN CENTRAL,URBANO,ECUADOR,11,2,1.310611e+09,12221.0,6029.0,6192.0,...,0.0,1.0,737.0,5384.0,354.0,105.0,234.0,1.0,274.0,"MULTIPOLYGON (((-70.70569 -33.45609, -70.70562..."


## 7. Controles de calidad

In [7]:
datos = ec_estudio.drop(columns=ec_estudio.geometry.name)
conteos = [c for c in datos.select_dtypes("number").columns if c.startswith("n_")]

print("Nulos en las variables elegidas:", int(datos[VARIABLES_ESTUDIO].isna().sum().sum()))
print("Valores negativos:", int((datos[conteos] < 0).sum().sum()))
print("Zonas repetidas (ID_ZONA):", int(datos["ID_ZONA"].duplicated().sum()))

print("\nTotales de Estación Central:")
print("  Personas (n_per):", int(datos["n_per"].sum()))
print("  Hogares (n_hog):", int(datos["n_hog"].sum()))
print("  Viviendas particulares (n_vp):", int(datos["n_vp"].sum()))

# Coherencia interna (la diferencia debería ser 0 o muy pequeña).
print("\nControles de coherencia:")
print("  n_per - (hombres + mujeres):", int((datos["n_per"] - datos["n_hombres"] - datos["n_mujeres"]).abs().sum()))
print("  n_vp - (ocupadas + desocupadas):", int((datos["n_vp"] - datos["n_vp_ocupada"] - datos["n_vp_desocupada"]).abs().sum()))

# Superficie de las zonas en km² (UTM zona 19S, EPSG:32719).
superficie_km2 = ec_estudio.to_crs(32719).area.sum() / 1e6
print(f"\nSuperficie cubierta por las zonas: {superficie_km2:.2f} km²")

print("\nSuma de cada conteo:")
datos[conteos].sum().astype(int)

Nulos en las variables elegidas: 0
Valores negativos: 0
Zonas repetidas (ID_ZONA): 0

Totales de Estación Central:
  Personas (n_per): 179223
  Hogares (n_hog): 67036
  Viviendas particulares (n_vp): 75310

Controles de coherencia:
  n_per - (hombres + mujeres): 0
  n_vp - (ocupadas + desocupadas): 0

Superficie cubierta por las zonas: 14.35 km²

Suma de cada conteo:


n_per                              179223
n_hombres                           87560
n_mujeres                           91663
n_hog                               67036
n_vp                                75310
n_vp_ocupada                        68313
n_vp_desocupada                      6997
n_tipo_viv_casa                     25826
n_tipo_viv_depto                    45741
n_tipo_viv_indigena                    19
n_tipo_viv_pieza                     3476
n_tipo_viv_mediagua                    60
n_tipo_viv_movil                        4
n_tipo_viv_otro                       184
n_comb_cocina_gas                   41381
n_comb_cocina_parafina                 37
n_comb_cocina_lena                     22
n_comb_cocina_pellet                    4
n_comb_cocina_carbon                   11
n_comb_cocina_electricidad          25277
n_comb_cocina_solar                     7
n_comb_cocina_no_utiliza              289
n_comb_calefaccion_gas              17642
n_comb_calefaccion_parafina       

**Para comparar:** el total de personas que resulte aquí debe coincidir (o ser muy parecido) con la cifra oficial del INE para Estación Central. Ese número es el que pide la calculadora de Huella Chile al crear el inventario, y no se puede editar después.

## 8. Guardar el resultado

In [8]:
RUTA_TOTALES_RM = "../IDG/totales_RM_censo2024.csv"
SALIDA_PARQUET = "../IDG/Parquet/censo2024_EstacionCentral.parquet"
SALIDA_CSV = "../IDG/censo2024_EstacionCentral.csv"


ec_estudio.to_parquet(SALIDA_PARQUET)
ec_estudio.drop(columns=ec_estudio.geometry.name).to_csv(SALIDA_CSV, index=False, encoding="utf-8-sig")
print("Guardado:", SALIDA_PARQUET, "y", SALIDA_CSV)

## 9. Totales de la Región Metropolitana (ejecutar una sola vez)

La base de Estación Central no trae los totales de la región, y se necesitan para calcular la participación de la comuna. Esta celda se ejecuta **una sola vez con la base completa** (`censo_zonal`) y guarda los totales de la RM en un archivo pequeño. Después ya no hace falta volver a cargar la base grande.

In [9]:
conteos_rm = [c for c in VARIABLES_ESTUDIO if c.startswith("n_")]

totales_rm = censo_zonal[conteos_rm].sum().to_frame("Región Metropolitana")
totales_rm.to_csv(RUTA_TOTALES_RM, encoding="utf-8-sig")
totales_rm.head()

,Región Metropolitana
n_per,7013927.0
n_hombres,3384006.0
n_mujeres,3629921.0
n_hog,2468960.0
n_vp,2607750.0


## 10. Participación de Estación Central en la Región Metropolitana

Sirve para repartir los datos regionales a la comuna (energía, transporte) y para conocer la distribución dentro de la comuna (aguas residuales, residuos): para cada combustible, el consumo de la comuna se estima como el consumo regional multiplicado por la participación de Estación Central en los hogares que lo usan.

Es el mismo enfoque del inventario 2022 (que usó el Censo 2017), actualizado con el Censo 2024. Supone que un hogar consume lo mismo en cualquier comuna para un mismo combustible.

Esta sección usa la base de Estación Central (`ec_estudio`) y el archivo de totales de la RM.

In [10]:
RUTA_EC = "../IDG/Parquet/censo2024_EstacionCentral.parquet"

ec_estudio = gpd.read_parquet(RUTA_EC)
totales_rm = pd.read_csv(RUTA_TOTALES_RM, index_col=0)["Región Metropolitana"]

COLS_COCINA = [c for c in VARIABLES_ESTUDIO if c.startswith("n_comb_cocina_")]
COLS_CALEF = [c for c in VARIABLES_ESTUDIO if c.startswith("n_comb_calefaccion_")]


def tabla_participacion(columnas):
    t = pd.DataFrame({
        "Estación Central": ec_estudio[columnas].sum(),
        "Región Metropolitana": totales_rm[columnas],
    })
    t["% de la RM que está en Estación Central"] = t["Estación Central"] / t["Región Metropolitana"] * 100
    t["% dentro de Estación Central"] = t["Estación Central"] / t["Estación Central"].sum() * 100
    return t.round(2)


# Un grupo por tema: cada uno se calcula igual y se guarda por separado.
GRUPOS = {
    "cocina": COLS_COCINA,
    "calefaccion": COLS_CALEF,
    "fuente_electricidad": [c for c in VARIABLES_ESTUDIO if c.startswith("n_fuente_elect_")],
    "basura": [c for c in VARIABLES_ESTUDIO if c.startswith("n_basura_")],
    "servicio_higienico": [c for c in VARIABLES_ESTUDIO if c.startswith("n_serv_hig_")],
    "transporte": [c for c in VARIABLES_ESTUDIO if c.startswith("n_transporte_")],
    "tipo_vivienda": [c for c in VARIABLES_ESTUDIO if c.startswith("n_tipo_viv_")],
}
participaciones = {nombre: tabla_participacion(cols) for nombre, cols in GRUPOS.items()}

# Se muestran primero los dos de energía; los demás quedan en el diccionario participaciones.
for nombre in ("cocina", "calefaccion"):
    print(f"== {nombre}")
    display(participaciones[nombre])

== cocina


,Estación Central,Región Metropolitana,% de la RM que está en Estación Central,% dentro de Estación Central
n_comb_cocina_gas,41381.0,2140248.0,1.93,61.74
n_comb_cocina_parafina,37.0,1466.0,2.52,0.06
n_comb_cocina_lena,22.0,1837.0,1.20,0.03
n_comb_cocina_pellet,4.0,109.0,3.67,0.01
n_comb_cocina_carbon,11.0,354.0,3.11,0.02
n_comb_cocina_electricidad,25277.0,315632.0,8.01,37.71
n_comb_cocina_solar,7.0,266.0,2.63,0.01
n_comb_cocina_no_utiliza,289.0,8720.0,3.31,0.43


== calefaccion


,Estación Central,Región Metropolitana,% de la RM que está en Estación Central,% dentro de Estación Central
n_comb_calefaccion_gas,17642.0,941315.0,1.87,26.32
n_comb_calefaccion_parafina,6720.0,392572.0,1.71,10.03
n_comb_calefaccion_lena,44.0,29575.0,0.15,0.07
n_comb_calefaccion_pellet,25.0,4182.0,0.60,0.04
n_comb_calefaccion_carbon,29.0,1507.0,1.92,0.04
n_comb_calefaccion_electricidad,26571.0,739173.0,3.59,39.64
n_comb_calefaccion_otra,270.0,13042.0,2.07,0.40
n_comb_calefaccion_no_utiliza,15722.0,347162.0,4.53,23.46


In [11]:
# ¿Cada conteo corresponde a hogares, viviendas o personas?
# Se compara la suma de cada tema con n_hog, n_vp_ocupada, n_vp y n_per.
bases = {
    "hogares (n_hog)": ec_estudio["n_hog"].sum(),
    "viviendas ocupadas (n_vp_ocupada)": ec_estudio["n_vp_ocupada"].sum(),
    "viviendas (n_vp)": ec_estudio["n_vp"].sum(),
    "personas (n_per)": ec_estudio["n_per"].sum(),
}
print("Totales de referencia:", {k: f"{v:,.0f}" for k, v in bases.items()}, "\n")

for nombre, cols in GRUPOS.items():
    total = ec_estudio[cols].sum().sum()
    cercana = min(bases, key=lambda k: abs(bases[k] - total))
    dif = (total - bases[cercana]) / bases[cercana] * 100
    print(f"{nombre:22s} suma = {total:>9,.0f} | se parece a {cercana} (diferencia {dif:+.2f} %)")

Totales de referencia: {'hogares (n_hog)': '67,036', 'viviendas ocupadas (n_vp_ocupada)': '68,313', 'viviendas (n_vp)': '75,310', 'personas (n_per)': '179,223'} 

cocina                 suma =    67,028 | se parece a hogares (n_hog) (diferencia -0.01 %)
calefaccion            suma =    67,023 | se parece a hogares (n_hog) (diferencia -0.02 %)
fuente_electricidad    suma =    64,316 | se parece a hogares (n_hog) (diferencia -4.06 %)
basura                 suma =    64,324 | se parece a hogares (n_hog) (diferencia -4.05 %)
servicio_higienico     suma =    64,329 | se parece a hogares (n_hog) (diferencia -4.04 %)
transporte             suma =    85,325 | se parece a viviendas (n_vp) (diferencia +13.30 %)
tipo_vivienda          suma =    75,310 | se parece a viviendas (n_vp) (diferencia +0.00 %)


In [12]:
# Guardar todas las tablas (una por tema) y una tabla consolidada.
for nombre, tabla in participaciones.items():
    tabla.to_csv(f"../IDG/participacion_{nombre}_EC_RM.csv", encoding="utf-8-sig")

consolidada = pd.concat(participaciones, names=["tema", "variable"])
consolidada.to_csv("../IDG/participacion_todas_EC_RM.csv", encoding="utf-8-sig")
consolidada

Estación Central  \
tema                variable                                            
cocina              n_comb_cocina_gas                         41381.0   
                    n_comb_cocina_parafina                       37.0   
                    n_comb_cocina_lena                           22.0   
                    n_comb_cocina_pellet                          4.0   
                    n_comb_cocina_carbon                         11.0   
                    n_comb_cocina_electricidad                25277.0   
                    n_comb_cocina_solar                           7.0   
                    n_comb_cocina_no_utiliza                    289.0   
calefaccion         n_comb_calefaccion_gas                    17642.0   
                    n_comb_calefaccion_parafina                6720.0   
                    n_comb_calefaccion_lena                      44.0   
                    n_comb_calefaccion_pellet                    25.0   
                    n_comb_calefaccion_carbon                    29.0   
                    n_comb_calefaccion_electricidad           26571.0   
                    n_comb_calefaccion_otra                     270.0   
                    n_comb_calefaccion_no_utiliza             15722.0   
fuente_electricidad n_fuente_elect_publica                    64141.0   
                    n_fuente_elect_diesel                        71.0   
                    n_fuente_elect_solar                         20.0   
                    n_fuente_elect_eolica                         9.0   
                    n_fuente_elect_otro                          46.0   
                    n_fuente_elect_no_tiene                      29.0   
basura              n_basura_servicios                        64121.0   
                    n_basura_entierra                            48.0   
                    n_basura_eriazo                              29.0   
                    n_basura_rio                                 14.0   
                    n_basura_otro                               112.0   
servicio_higienico  n_serv_hig_alc_dentro                     61612.0   
                    n_serv_hig_alc_fuera                       2634.0   
                    n_serv_hig_fosa                              41.0   
                    n_serv_hig_pozo                              18.0   
                    n_serv_hig_acequia_canal                      4.0   
                    n_serv_hig_cajon_otro                         0.0   
                    n_serv_hig_bano_quimico                       2.0   
                    n_serv_hig_bano_seco                          0.0   
                    n_serv_hig_no_tiene                          18.0   
transporte          n_transporte_auto                         13541.0   
                    n_transporte_publico                      58574.0   
                    n_transporte_camina                        6668.0   
                    n_transporte_bicicleta                     1946.0   
                    n_transporte_motocicleta                   1865.0   
                    n_transporte_cab_lan_bote                     8.0   
                    n_transporte_otros                         2723.0   
tipo_vivienda       n_tipo_viv_casa                           25826.0   
                    n_tipo_viv_depto                          45741.0   
                    n_tipo_viv_indigena                          19.0   
                    n_tipo_viv_pieza                           3476.0   
                    n_tipo_viv_mediagua                          60.0   
                    n_tipo_viv_movil                              4.0   
                    n_tipo_viv_otro                             184.0   

                                                     Región Metropolitana  \
tema                variable                                                
cocina              n_comb_cocina_gas                           2140248.0   
                    n_com